# 01 — Data Ingestion & Quality Control

**Pipeline Step 1 of 5**

This notebook downloads **6 Vehicle-only spatial transcriptomics samples** from GEO [GSE319409](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE319409) (*Mapping the Secondary Response to Traumatic Brain Injury Using Spatial Transcriptomics*) and applies a standard QC workflow before saving the preprocessed data for downstream analysis.

**Study design (Vehicle-only subset):**
| Group | N | GEO Accessions |
|-------|---|----------------|
| Sham + Vehicle | 3 | GSM9517469, GSM9517475, GSM9517481 |
| TBI + Vehicle | 3 | GSM9517472, GSM9517477, GSM9517478 |

The platform is **10x Visium CytAssist** (FFPE coronal mouse brain sections, 7 days post-injury, Space Ranger v2.1.1). Each sample contains ~3,600–4,400 capture spots profiling ~19,400 genes with spatial coordinates and tissue images.

### Outputs
| File | Description |
|---|---|
| `data/processed/tbi_preprocessed.h5ad` | QC-filtered, normalized AnnData ready for feature selection |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_ingestion import get_dataset
from src.spatial_pipeline import load_adata, run_qc, normalize

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Imports ready.")

Imports ready.


## 1.1 Download Dataset

We download the 6 Vehicle-only samples from GEO GSE319409. For each sample the pipeline fetches the 10x MEX matrix files (barcodes, features, matrix), spatial coordinates (CytAssist `tissue_positions.csv`), scale factors, and tissue images from the GEO FTP server.

The data is cached locally as a single concatenated `.h5ad` file so re-runs skip the download step. Per-sample metadata (`sample_id`, `condition`, `treatment`, `batch`) is embedded in `adata.obs`.

In [2]:
h5ad_path = get_dataset(DATA_RAW, source="geo_tbi")
adata = load_adata(h5ad_path)

print(f"\nRaw dataset: {adata.shape[0]} spots × {adata.shape[1]} genes")
print(f"Samples: {adata.obs['sample_id'].nunique()}")
print(f"Conditions: {dict(adata.obs['condition'].value_counts())}")

  GEO TBI dataset already cached: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/tbi_geo.h5ad
  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/tbi_geo.h5ad
  Shape: 24940 spots × 19465 genes

Raw dataset: 24940 spots × 19465 genes
Samples: 6
Conditions: {0: np.int64(12759), 1: np.int64(12181)}


## 1.2 Quality Control

Standard QC filters applied:

1. **Minimum genes per spot (≥ 200):** Remove empty or damaged spots.
2. **Minimum cells per gene (≥ 3):** Exclude rarely detected genes.
3. **Maximum mitochondrial gene percentage (< 30%):** Remove stressed/dying spots. Brain tissue naturally has higher mitochondrial fractions due to the high metabolic demand of neurons, so we use a relaxed threshold of 30%.

In [3]:
adata = run_qc(adata)
print(f"\nPost-QC: {adata.shape[0]} spots × {adata.shape[1]} genes")

  QC filtering: 24940 → 24922 spots
  Genes retained: 19415

Post-QC: 24922 spots × 19415 genes


## 1.3 Normalization

After QC filtering, we apply two normalization steps:

1. **Library-size normalization (CPM-like):** Each spot's total UMI count is scaled to a common target of 10,000 counts. This corrects for variation in sequencing depth between spots, making expression values comparable.
2. **Log-transformation (`log1p`):** A `log(x + 1)` transform is applied to stabilize variance and compress the dynamic range. Highly expressed genes (e.g., ribosomal, mitochondrial) would otherwise dominate variance-based analyses.

Raw counts are preserved in `adata.raw` so that they remain available for differential expression and spatial plotting (which expect un-normalized or separately normalized values).

In [4]:
adata = normalize(adata)
print(f"\nNormalized: {adata.shape[0]} spots × {adata.shape[1]} genes")
print(f"Raw layer preserved: {adata.raw is not None}")

  Normalized to 10000 counts/cell and log1p-transformed

Normalized: 24922 spots × 19415 genes
Raw layer preserved: True


## 1.4 Save Preprocessed Data

The preprocessed AnnData is serialized to `.h5ad` format. Downstream notebooks (Stabl, visualization, KG) load this checkpoint directly.

In [5]:
out_path = DATA_PROCESSED / "tbi_preprocessed.h5ad"
adata.write_h5ad(out_path)
print(f"Saved preprocessed AnnData to {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

Saved preprocessed AnnData to /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/tbi_preprocessed.h5ad
File size: 3746.5 MB
